<a href="https://colab.research.google.com/github/BillJr99/Ursinus-CS357-Fall2026/blob/gh-pages/files/notebooks/MonteCarloRetirement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Monte Carlo Retirement Simulation: Multimodal Analysis and Tool Calling

## What This Notebook Demonstrates

This notebook is the runnable companion to the **Multimodal AI and Monte Carlo Simulation lab** (`/Assignments/MonteCarlo`). It walks the same path as the lab, in executable cells:

1. **Simulation**: a 1,000-path Monte Carlo retirement simulation with NumPy, drawing annual returns from a configurable normal distribution.
2. **Visualization**: a labeled two-panel chart (portfolio paths with median and percentile bands; histogram of final balances) saved as a PNG.
3. **Ground truth**: a `simulation_stats.txt` file with the exact statistics the chart encodes.
4. **Multimodal analysis**: the PNG, base64-encoded, sent to a local vision model (llava) via the Ollama `/api/generate` endpoint, with a two-turn conversation about the chart.
5. **Comparison**: the AI's numeric claims checked against the ground-truth statistics.
6. **Tool calling (new)**: the simulation wrapped as a `run_retirement_sim` tool with a JSON schema, driven by an agent loop in which the model *chooses the parameters*, invokes the tool, and then interprets the resulting chart.

**Course connection:** this notebook bridges two agentic capabilities. Parts 1-5 treat the model as an *interpreter* of an artifact you made; Part 6 promotes it to an *actor* that decides what experiment to run. That promotion (from reading outputs to choosing inputs) is exactly what the Tool Use and Function Calling session is about, and it raises the evaluation question this lab keeps returning to: when the agent both picks the parameters and narrates the results, who checks its numbers?

## Setup

Install the (few) dependencies. Everything in Parts 1-3 runs anywhere, including plain Colab. Parts 4-6 call a **local Ollama server** and will fail on plain Colab; canned sample responses are provided so you can complete the analysis offline.

In [ ]:
!pip install numpy matplotlib requests

In [ ]:
import base64
import io
import json
import os
import traceback

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests

## Part 1: The Simulation Engine

A spreadsheet gives you one future; Monte Carlo gives you a thousand. Each simulated year draws a random annual return from a normal distribution, applies it to the portfolio after that year's contributions, and records the balance. The configuration mirrors the lab's `config.json`.

In [ ]:
CFG = {
    "starting_age": 25,
    "retirement_age": 65,
    "life_expectancy": 90,
    "starting_savings": 10000,
    "monthly_contribution": 500,
    "annual_return_mean": 0.07,
    "annual_return_std": 0.12,
    "inflation_rate": 0.025,
    "num_simulations": 1000,
    "model": "llava",            # vision model for Parts 4-5
    "tool_model": "llama3.1",    # tool-calling model for Part 6
    "ollama_url": "http://localhost:11434",
}

# Write it out as config.json too, so this notebook matches the lab's file layout.
with open("config.json", "w") as f:
    json.dump(CFG, f, indent=2)
print("Wrote config.json")

In [ ]:
def simulate_retirement(cfg):
    """
    Run a Monte Carlo retirement simulation.

    Draws annual returns from N(annual_return_mean, annual_return_std) for each
    simulated year of a career, applying monthly contributions each year.

    Returns:
        np.ndarray of shape (num_simulations, years_to_retirement), where each
        row is one simulated portfolio path and each column is the
        end-of-year balance.
    """
    years = cfg["retirement_age"] - cfg["starting_age"]
    results = np.zeros((cfg["num_simulations"], years))

    for sim in range(cfg["num_simulations"]):
        balance = cfg["starting_savings"]
        for year in range(years):
            # Add this year's contributions
            balance += cfg["monthly_contribution"] * 12

            # Draw a random annual return and apply it
            annual_return = np.random.normal(
                cfg["annual_return_mean"], cfg["annual_return_std"]
            )
            balance *= (1 + annual_return)

            # A portfolio cannot go negative
            balance = max(balance, 0)

            results[sim, year] = balance

    return results


np.random.seed(42)
balances = simulate_retirement(CFG)
print("Simulated paths:", balances.shape)
print(f"Median final balance: ${np.median(balances[:, -1]):,.0f}")

### YOUR TURN: Volatility and the Fan

**Before running: what do you expect and why?** Keep the mean return at 7% but double the volatility (`annual_return_std` from 0.12 to 0.24). Predict, before running: will the **median** final balance go up, down, or stay about the same? What about the **10th percentile**? (Careful: the answers are different, and the reason is the asymmetry of compounding.)

In [ ]:
# YOUR TURN: fill in the blanks (___) to run a high-volatility scenario
# and compare it to the baseline above.

cfg_volatile = {**CFG, "annual_return_std": ___}   # fill in: double the baseline std

np.random.seed(42)
balances_volatile = simulate_retirement(cfg_volatile)
final_v = balances_volatile[:, -1]
final_b = balances[:, -1]

print(f"{'':<18}{'Baseline':>16}{'High volatility':>18}")
print(f"{'Median':<18}${np.median(final_b):>15,.0f}${np.median(final_v):>17,.0f}")
# Fill in: the percentile to examine the WORST-case outcomes (a small number, not 50)
print(f"{'10th percentile':<18}${np.percentile(final_b, ___):>15,.0f}${np.percentile(final_v, ___):>17,.0f}")

# After running: did the median move the way you predicted? Did the 10th percentile?

## Part 2: Visualization

The two-panel chart is the artifact everything else in this notebook analyzes: (left) all 1,000 paths with the median and 10th/90th percentile bands; (right) the histogram of final balances with the median and the $1M milestone marked. The function saves `retirement_simulation.png` and returns the base64-encoded PNG string we will send to the vision model.

In [ ]:
def plot_simulation(balances, cfg, filename="retirement_simulation.png"):
    """
    Create a labeled two-panel visualization of simulation results.

    Saves the figure to `filename` and returns a base64-encoded PNG string
    for sending to the multimodal model.
    """
    ages = list(range(cfg["starting_age"] + 1, cfg["retirement_age"] + 1))
    median_path = np.median(balances, axis=0)
    p10_path = np.percentile(balances, 10, axis=0)
    p90_path = np.percentile(balances, 90, axis=0)
    final_balances = balances[:, -1]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # Left panel: portfolio paths
    for path in balances:
        ax1.plot(ages, path, color="gray", alpha=0.05, linewidth=0.5)
    ax1.plot(ages, median_path, color="blue", linewidth=2, label="Median")
    ax1.plot(ages, p10_path, color="red", linewidth=1.5, linestyle="--",
             label="10th / 90th percentile")
    ax1.plot(ages, p90_path, color="red", linewidth=1.5, linestyle="--")
    ax1.set_xlabel("Age", fontsize=12)
    ax1.set_ylabel("Portfolio Balance", fontsize=12)
    ax1.set_title(
        f"Monte Carlo Retirement Simulation\n"
        f"{cfg['num_simulations']:,} paths | "
        f"${cfg['monthly_contribution']:,}/mo contribution | "
        f"Mean return {cfg['annual_return_mean']:.0%}",
        fontsize=11,
    )
    ax1.legend(fontsize=10)
    ax1.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x:,.0f}")
    )

    # Right panel: final balance histogram
    ax2.hist(final_balances, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
    ax2.axvline(np.median(final_balances), color="blue", linewidth=2,
                label=f"Median: ${np.median(final_balances):,.0f}")
    ax2.axvline(1_000_000, color="green", linewidth=1.5, linestyle="--",
                label="$1 Million milestone")
    ax2.set_xlabel("Final Balance at Retirement", fontsize=12)
    ax2.set_ylabel("Number of Simulations", fontsize=12)
    ax2.set_title(f"Distribution of Final Balances at Age {cfg['retirement_age']}", fontsize=11)
    ax2.legend(fontsize=10)
    ax2.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M")
    )

    plt.tight_layout()
    fig.savefig(filename, dpi=150, bbox_inches="tight")
    print(f"Saved: {filename}")

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


image_b64 = plot_simulation(balances, CFG)
print(f"Base64 PNG length: {len(image_b64):,} characters")

## Part 3: Ground-Truth Statistics

Before asking any AI to read the chart, we compute the exact numbers ourselves and save them to `simulation_stats.txt`. This file is the **ground truth** for every comparison that follows: in this lab, *you* are the evaluator, and this is your answer key.

In [ ]:
def save_statistics(balances, cfg, path="simulation_stats.txt"):
    """Compute and save key statistics from the simulation to a text file."""
    final = balances[:, -1]
    prob_million = (final >= 1_000_000).mean()

    lines = [
        "Simulation parameters:",
        f"  Monthly contribution: ${cfg['monthly_contribution']:,}",
        f"  Mean annual return:   {cfg['annual_return_mean']:.1%}",
        f"  Return std dev:       {cfg['annual_return_std']:.1%}",
        f"  Number of paths:      {cfg['num_simulations']:,}",
        "",
        f"Final balance at retirement (age {cfg['retirement_age']}):",
        f"  10th percentile: ${np.percentile(final, 10):>12,.0f}",
        f"  Median (50th):   ${np.median(final):>12,.0f}",
        f"  90th percentile: ${np.percentile(final, 90):>12,.0f}",
        f"  Mean:            ${final.mean():>12,.0f}",
        "",
        f"Probability of reaching $1 million: {prob_million:.1%}",
    ]

    text = "\n".join(lines)
    print(text)
    with open(path, "w") as f:
        f.write(text)
    print(f"\nSaved: {path}")
    return text


stats_text = save_statistics(balances, CFG)
TRUE_PROB_MILLION = (balances[:, -1] >= 1_000_000).mean()

## Part 4: Asking a Multimodal Model to Read the Chart

> **This section requires a local Ollama server** (`ollama pull llava`, then verify `curl http://localhost:11434/api/tags`). **It will fail on plain Colab**, which cannot reach your machine's localhost. If you are offline or on Colab, set `USE_OLLAMA = False` below and the notebook substitutes a canned sample response, deliberately written with the kind of confident-but-imprecise number a real vision model produces, so every later cell still works.

We send the base64 PNG in the `images` array of an Ollama `/api/generate` request. Turn 1 uses a structured prompt (role, four required sections, stated audience); Turn 2 presses the model on its quantitative claim.

In [ ]:
USE_OLLAMA = False   # set True if you have a local Ollama server with llava pulled

# Canned responses for offline / Colab work. These mimic real llava output:
# fluent, structured, pattern-level observations that are largely right --
# and a specific percentage that is fabricated from visual impression.
SAMPLE_TURN1 = (
    "1. The gap between the red dashed lines shows that outcomes diverge "
    "enormously over 40 years: disciplined savers with identical habits can end "
    "up with very different balances purely due to market luck. The spread "
    "widens with age because returns compound.\n"
    "2. Based on the histogram, approximately 68% of simulations ended above "
    "$1 million by retirement.\n"
    "3. Start contributing as early as possible: the fan is narrow in the early "
    "years, so increasing contributions now shifts the entire distribution up "
    "later, including the worst-case outcomes.\n"
    "4. The simulation assumes returns are drawn independently each year from a "
    "normal distribution, which understates the chance of multi-year crashes; a "
    "deployer should disclose that real markets have correlated bad years."
)
SAMPLE_TURN2 = (
    "Looking at the right panel, the green dashed $1 million line appears to "
    "fall left of the histogram's main mass, and the bars to its right look "
    "taller and more numerous. I estimated roughly two-thirds of the area lies "
    "to the right of the line, which is how I arrived at 68%. I am fairly "
    "confident in that figure, though reading exact areas from a histogram "
    "image is imprecise."
)


def ask_multimodal_model(image_b64, question, cfg):
    """
    Send an image and a text question to a local multimodal model via Ollama.

    Returns the model's text response (or raises after printing a traceback).
    """
    payload = {
        "model": cfg["model"],
        "prompt": question,
        "images": [image_b64],
        "stream": False,
    }
    try:
        url = cfg["ollama_url"] + "/api/generate"
        response = requests.post(url, json=payload, timeout=120)
        response.raise_for_status()
        return response.json()["response"]
    except Exception as e:
        print(f"[ask_multimodal_model] {e}")
        traceback.print_exc()
        raise

In [ ]:
initial_prompt = (
    "You are a financial educator analyzing a Monte Carlo retirement simulation "
    "chart for a college student audience with no prior finance background.\n\n"
    "The chart has two panels:\n"
    "- Left panel: 1,000 simulated portfolio paths from age 25 to 65, with a "
    "solid blue median line and two red dashed lines showing the 10th and 90th "
    "percentile bounds.\n"
    "- Right panel: a histogram of final portfolio balances at age 65, with a "
    "vertical blue line at the median and a vertical green dashed line at $1 million.\n\n"
    "Please analyze the chart and respond in exactly four numbered sections:\n"
    "1. What the spread of paths (the gap between the dashed red lines) tells us "
    "about retirement savings risk.\n"
    "2. Your estimate of what percentage of simulations ended above $1 million, "
    "based on the histogram.\n"
    "3. One specific, actionable insight for a 25-year-old starting their career.\n"
    "4. One limitation of this simulation that a tool deployer should disclose to users."
)

followup = (
    "Based specifically on the histogram in the right panel, walk me through your "
    "reasoning for the percentage estimate you gave in section 2. What visual "
    "features of the histogram did you use, and how confident are you in that number?"
)

if USE_OLLAMA:
    print("=== Turn 1: Initial Analysis ===")
    response1 = ask_multimodal_model(image_b64, initial_prompt, CFG)
    print(response1)
    print("\n=== Turn 2: Follow-Up ===")
    response2 = ask_multimodal_model(image_b64, followup, CFG)
    print(response2)
else:
    print("[offline mode] Using canned sample responses.\n")
    response1, response2 = SAMPLE_TURN1, SAMPLE_TURN2
    print("=== Turn 1 (sample) ===\n" + response1)
    print("\n=== Turn 2 (sample) ===\n" + response2)

with open("model_responses.txt", "w") as f:
    f.write("=== Turn 1 ===\n" + response1 + "\n\n=== Turn 2 ===\n" + response2 + "\n")
print("\nSaved: model_responses.txt")

## Part 5: Comparing AI Claims to Ground Truth

### YOUR TURN: Check the Number

**Before running: what do you expect and why?** The model claimed a specific percentage of simulations ended above $1 million (68% in the canned response). Your `simulation_stats.txt` holds the true value. Before running the cell: do you expect the model's estimate to be within 5 percentage points of the truth? What visual feature of a right-skewed histogram makes "area to the right of a line" hard to eyeball?

In [ ]:
# YOUR TURN: fill in the blanks (___) to compare the AI's claim to ground truth.

ai_claimed_percent = ___      # fill in: the percentage the model stated (as a number, e.g. 68)
true_percent = TRUE_PROB_MILLION * 100

error = abs(ai_claimed_percent - true_percent)

print(f"AI claim:     {ai_claimed_percent:.1f}% of simulations ended above $1M")
print(f"Ground truth: {true_percent:.1f}%  (from simulation_stats.txt)")
print(f"Absolute error: {error:.1f} percentage points")

# Fill in: a threshold (in percentage points) beyond which you would call the
# claim 'wrong' rather than 'approximately correct' -- and defend it in a comment.
verdict = "wrong" if error > ___ else "approximately correct"
print(f"Verdict: {verdict}")

# For your writeup: copy the model's exact sentence from model_responses.txt
# alongside the true value. Verbatim excerpt + ground truth is the lab's standard
# of evidence for every claim about AI behavior.

## Part 6: The Simulation as a Tool, Function Calling

So far the model only *interpreted* an experiment we designed. Now we invert the relationship: we wrap the simulation as a **tool**, hand the model its JSON schema, and let the model **choose the parameters**, invoke the tool, and interpret the chart it asked for.

> **This section also requires a local Ollama server**, with a *tool-capable* model (`ollama pull llama3.1`, note that llava does not support function calling; we use llava only for the vision turn). With `USE_OLLAMA = False`, a canned tool call and interpretation are substituted so the flow still runs end to end.

First, the tool itself. Note the new `stock_allocation` parameter: it blends an equity-like return distribution with a bond-like one, giving the agent a genuinely meaningful lever to reason about.

In [ ]:
def run_retirement_sim(years=40, annual_contribution=6000, stock_allocation=0.8,
                       n_paths=1000, seed=42):
    """
    Run a Monte Carlo retirement simulation and return summary statistics
    plus the path to a saved two-panel chart.

    Args:
        years: number of years of saving (career length).
        annual_contribution: dollars contributed per year.
        stock_allocation: fraction 0.0-1.0 of the portfolio in stocks; the
            remainder is in bonds. Stocks: mean 8%, std 15%. Bonds: mean 3%,
            std 5%. Blended linearly.
        n_paths: number of Monte Carlo paths.
        seed: random seed for reproducibility.

    Returns:
        dict with keys: median, p10, p90, mean, prob_million, years,
        annual_contribution, stock_allocation, n_paths, seed, chart_path.
    """
    stock_allocation = float(np.clip(stock_allocation, 0.0, 1.0))
    mean_return = stock_allocation * 0.08 + (1 - stock_allocation) * 0.03
    std_return = stock_allocation * 0.15 + (1 - stock_allocation) * 0.05

    cfg = {
        "starting_age": 25,
        "retirement_age": 25 + int(years),
        "starting_savings": 10000,
        "monthly_contribution": annual_contribution / 12,
        "annual_return_mean": mean_return,
        "annual_return_std": std_return,
        "num_simulations": int(n_paths),
    }

    np.random.seed(int(seed))
    sim_balances = simulate_retirement(cfg)
    chart_path = "tool_call_simulation.png"
    tool_image_b64 = plot_simulation(sim_balances, cfg, filename=chart_path)

    final = sim_balances[:, -1]
    stats = {
        "median": round(float(np.median(final))),
        "p10": round(float(np.percentile(final, 10))),
        "p90": round(float(np.percentile(final, 90))),
        "mean": round(float(final.mean())),
        "prob_million": round(float((final >= 1_000_000).mean()), 3),
        "years": int(years),
        "annual_contribution": float(annual_contribution),
        "stock_allocation": stock_allocation,
        "n_paths": int(n_paths),
        "seed": int(seed),
        "chart_path": chart_path,
    }
    # Stash the image for the vision turn
    stats["_image_b64"] = tool_image_b64
    return stats


# Smoke test the tool directly before handing it to an agent.
test_stats = run_retirement_sim(years=30, annual_contribution=6000,
                                stock_allocation=0.6, n_paths=500, seed=1)
print({k: v for k, v in test_stats.items() if not k.startswith("_")})

In [ ]:
# The JSON schema the model sees. This -- not your Python code -- is the tool's
# entire interface from the agent's point of view. Every design decision here
# (names, descriptions, bounds) shapes what parameters the model will choose.

RETIREMENT_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "run_retirement_sim",
        "description": (
            "Run a Monte Carlo retirement savings simulation and return summary "
            "statistics (median, 10th/90th percentile, mean, probability of "
            "reaching $1 million) plus a saved two-panel chart of the results."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "years": {
                    "type": "integer",
                    "description": "Number of years of saving before retirement (e.g., 40 for a full career starting at 25).",
                    "minimum": 1,
                    "maximum": 60,
                },
                "annual_contribution": {
                    "type": "number",
                    "description": "Dollars contributed per year (e.g., 6000).",
                    "minimum": 0,
                },
                "stock_allocation": {
                    "type": "number",
                    "description": "Fraction of the portfolio in stocks, 0.0-1.0. Higher values raise both expected return and volatility.",
                    "minimum": 0.0,
                    "maximum": 1.0,
                },
                "n_paths": {
                    "type": "integer",
                    "description": "Number of Monte Carlo simulation paths (1000 recommended).",
                    "minimum": 100,
                    "maximum": 10000,
                },
                "seed": {
                    "type": "integer",
                    "description": "Random seed for reproducibility.",
                },
            },
            "required": ["years", "annual_contribution", "stock_allocation"],
        },
    },
}

print(json.dumps(RETIREMENT_TOOL_SCHEMA, indent=2))

### The Agent Loop

The loop has three steps, and each is a distinct agentic capability:

1. **Choose**: we give a tool-capable model a user goal in plain English plus the schema, via Ollama's `/api/chat` endpoint with a `tools` array. The model responds not with prose but with a *tool call*: a function name and a JSON object of arguments it selected.
2. **Invoke**: *our code* (never the model) executes `run_retirement_sim` with those arguments. The model cannot run anything; it can only ask.
3. **Interpret**: we send the resulting chart to the vision model and the numeric stats back to the tool-calling model, and ask for a recommendation.

Watch what the model picks. Are the parameters reasonable for the stated goal? Defensible? That judgment is yours to make in the critique below.

In [ ]:
USER_GOAL = (
    "I am 25 and want to know whether contributing $500 a month with a fairly "
    "aggressive portfolio gives me a good chance of retiring at 65 with over "
    "$1 million. Choose appropriate simulation parameters, run the simulation, "
    "and then interpret the results for me."
)

# Canned tool call for offline mode -- the kind of choice llama3.1 typically makes.
SAMPLE_TOOL_CALL = {
    "name": "run_retirement_sim",
    "arguments": {"years": 40, "annual_contribution": 6000,
                  "stock_allocation": 0.8, "n_paths": 1000, "seed": 7},
}


def request_tool_call(goal, cfg):
    """Ask the tool-capable model to choose parameters via Ollama function calling."""
    payload = {
        "model": cfg["tool_model"],
        "messages": [
            {"role": "system",
             "content": "You are a retirement planning assistant. Use the provided "
                        "simulation tool to answer quantitative questions. Choose "
                        "parameters that faithfully reflect the user's stated situation."},
            {"role": "user", "content": goal},
        ],
        "tools": [RETIREMENT_TOOL_SCHEMA],
        "stream": False,
    }
    url = cfg["ollama_url"] + "/api/chat"
    response = requests.post(url, json=payload, timeout=120)
    response.raise_for_status()
    message = response.json()["message"]
    calls = message.get("tool_calls") or []
    if not calls:
        raise RuntimeError(f"Model did not call the tool. It said: {message.get('content')}")
    call = calls[0]["function"]
    return {"name": call["name"], "arguments": call["arguments"]}


if USE_OLLAMA:
    tool_call = request_tool_call(USER_GOAL, CFG)
else:
    print("[offline mode] Using canned sample tool call.\n")
    tool_call = SAMPLE_TOOL_CALL

print("Model chose tool:", tool_call["name"])
print("Model chose arguments:", json.dumps(tool_call["arguments"], indent=2))

# Step 2: OUR code invokes the tool with the model's chosen arguments.
agent_stats = run_retirement_sim(**tool_call["arguments"])
public_stats = {k: v for k, v in agent_stats.items() if not k.startswith("_")}
print("\nTool result:")
print(json.dumps(public_stats, indent=2))

with open("tool_call_transcript.txt", "w") as f:
    f.write("=== User goal ===\n" + USER_GOAL + "\n\n")
    f.write("=== Model tool call ===\n" + json.dumps(tool_call, indent=2) + "\n\n")
    f.write("=== Tool result ===\n" + json.dumps(public_stats, indent=2) + "\n")
print("\nSaved: tool_call_transcript.txt")

In [ ]:
# Step 3: Interpretation. The vision model reads the chart the AGENT asked for,
# and we log its narrative alongside the tool's exact numbers.

SAMPLE_INTERPRETATION = (
    "The simulation you requested shows a strong outcome: the median path ends "
    "well above the $1 million milestone, and roughly three-quarters of the "
    "simulated futures clear it. The downside band still matters -- the 10th "
    "percentile ends materially lower -- but with a 40-year horizon and an 80% "
    "stock allocation, your plan of about $500 a month puts the $1 million goal "
    "more likely than not. Consider revisiting the allocation as retirement nears."
)

interpretation_prompt = (
    "You chose the parameters for this Monte Carlo retirement simulation and the "
    "attached chart shows the result. The tool also returned these exact statistics: "
    + json.dumps(public_stats)
    + ". In 3-5 sentences for a 25-year-old with no finance background, interpret "
    "the chart and state whether the $1 million goal is more likely than not. "
    "Quote the tool's exact probability rather than estimating from the image."
)

if USE_OLLAMA:
    interpretation = ask_multimodal_model(agent_stats["_image_b64"],
                                          interpretation_prompt, CFG)
else:
    print("[offline mode] Using canned sample interpretation.\n")
    interpretation = SAMPLE_INTERPRETATION

print(interpretation)

with open("tool_call_transcript.txt", "a") as f:
    f.write("\n=== Model interpretation ===\n" + interpretation + "\n")
print("\nAppended interpretation to tool_call_transcript.txt")

### YOUR TURN: Critique the Agent

**Before running: what do you expect and why?** The user asked about "$500 a month"; did the model's chosen `annual_contribution` actually equal $500 × 12? Is an 0.8 stock allocation a faithful reading of "fairly aggressive"? And does the interpretation's language ("roughly three-quarters", "more likely than not") match the tool's exact `prob_million`? Predict which of the three will hold up before you check.

In [ ]:
# YOUR TURN: fill in the blanks (___) to audit the agent against ground truth.

chosen = tool_call["arguments"]

# Check 1: parameter fidelity. The user said $500/month.
expected_contribution = ___ * 12        # fill in: the monthly amount the user stated
print(f"Contribution -- user implied ${expected_contribution:,.0f}/yr, "
      f"model chose ${chosen['annual_contribution']:,.0f}/yr: "
      f"{'MATCH' if chosen['annual_contribution'] == expected_contribution else 'MISMATCH'}")

# Check 2: horizon fidelity. Age 25 to 65.
print(f"Years -- expected 40, model chose {chosen['years']}: "
      f"{'MATCH' if chosen['years'] == 40 else 'MISMATCH'}")

# Check 3: interpretation fidelity. Compare the narrative to the tool's number.
true_prob = public_stats[___]           # fill in: the stats key holding P(>$1M)
print(f"\nTool's exact probability of reaching $1M: {true_prob:.1%}")
print("Now re-read the interpretation above. Does its qualitative claim")
print("('roughly three-quarters', 'more likely than not', etc.) match this number?")

# For your writeup: record each check as correct / approximately correct / wrong,
# with the verbatim excerpt. The same evidence standard as Part 5 applies -- but
# now you are auditing BOTH the agent's inputs (parameter choices) and its
# outputs (interpretation).

## Wrap-Up

You have now seen the same simulation from both sides of the agentic divide:

- **Model as interpreter** (Parts 1-5): you designed the experiment; the model read the picture and got the pattern right but the number wrong.
- **Model as actor** (Part 6): the model designed the experiment through a JSON schema you wrote, your code executed it, and the model narrated a chart it had asked for.

The lab's closing question applies doubly here: an agent that chooses its own parameters *and* interprets its own results can quietly compound an early mistranslation (the wrong contribution, an unjustified allocation) into a confident final recommendation. The ground-truth statistics file (and the audit habit you practiced above) is the guardrail.

Continue in the lab writeup: `/Assignments/MonteCarlo`, Part 5 ("The Simulation as a Tool: Function-Calling Extension").